In [67]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
uri = pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1)
engine = create_engine(uri, future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)


/Users/medovuk/programming/python_projects/study/DataBase course/HW_2/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


## 1.2. Отримання даних

Датасет: NYC TLC Yellow Taxi Trip Records, січень 2024.

Джерело: офіційна сторінка TLC, файл `yellow_tripdata_2024-01.parquet`.

Вибірка: 11 колонок, 100 000 випадкових рядків (`random_state=42` для відтворюваності).

In [68]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

parquet_path = data_dir / 'yellow_tripdata_2024-01.parquet'
csv_path = data_dir / 'hw3_taxi_sample.csv'

if not parquet_path.exists():
    urlretrieve(DATA_URL, parquet_path)

columns = [
    'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID',
    'payment_type', 'fare_amount', 'tip_amount', 'total_amount', 'store_and_fwd_flag'
]

df_full = pd.read_parquet(parquet_path, columns=columns)
df = df_full.sample(n=100_000, random_state=42).sort_values('tpep_pickup_datetime')

df.to_csv(csv_path, index=False)

print('full month:', df_full.shape)
print('sample:   ', df.shape)
df.head()

full month: (2964624, 12)
sample:    (100000, 12)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,total_amount,store_and_fwd_flag
256,2,2023-12-31 23:56:46,2024-01-01 00:12:06,2.0,2.38,236,142,1,15.6,1.00,21.60,N
4137,2,2023-12-31 23:58:35,2024-01-01 00:13:06,6.0,8.39,138,217,2,33.1,0.00,42.35,N
3782,2,2024-01-01 00:00:17,2024-01-01 00:07:46,1.0,2.35,161,75,1,12.1,3.42,20.52,N
5138,2,2024-01-01 00:01:39,2024-01-01 01:02:53,1.0,2.59,161,246,2,47.1,0.00,52.10,N
2936,2,2024-01-01 00:02:35,2024-01-01 00:10:55,1.0,1.36,90,79,1,10.0,3.00,18.00,N


In [69]:
ddl_staging = """
DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;
DROP TABLE IF EXISTS hw3_taxi_staging CASCADE;

CREATE TABLE hw3_taxi_staging (
    vendor_id_raw        TEXT,
    pickup_raw           TEXT,
    dropoff_raw          TEXT,
    passenger_count_raw  TEXT,
    trip_distance_raw    TEXT,
    pu_location_id_raw   TEXT,
    do_location_id_raw   TEXT,
    payment_type_raw     TEXT,
    fare_amount_raw      TEXT,
    tip_amount_raw       TEXT,
    total_amount_raw     TEXT,
    store_and_fwd_flag_raw TEXT
);
"""

with engine.begin() as conn:
    conn.execute(text(ddl_staging))

In [70]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                """
                COPY hw3_taxi_staging (
                    vendor_id_raw, pickup_raw, dropoff_raw,
                    passenger_count_raw, trip_distance_raw,
                    pu_location_id_raw, do_location_id_raw,
                    payment_type_raw, fare_amount_raw,
                    tip_amount_raw, total_amount_raw,
                    store_and_fwd_flag_raw
                )
                FROM STDIN
                WITH (FORMAT csv, HEADER true, DELIMITER ',')
                """,
                file
            )
    raw_conn.commit()
finally:
    raw_conn.close()

pd.read_sql('SELECT COUNT(*) AS loaded_rows FROM hw3_taxi_staging', engine)

,loaded_rows
0,100000


In [71]:
pd.read_sql("""
SELECT passenger_count_raw, COUNT(*) AS n
FROM hw3_taxi_staging
GROUP BY passenger_count_raw
ORDER BY n DESC
""", engine)

,passenger_count_raw,n
0,1.0,73739
1,2.0,13809
2,NaN,4727
3,3.0,3071
4,4.0,1722
5,5.0,1140
6,0.0,1085
7,6.0,705
8,8.0,2


In [72]:
ddl_clean = """
DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;

CREATE TABLE hw3_taxi_trips (
    trip_id              BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,

    vendor_id            SMALLINT NOT NULL,

    pickup_at            TIMESTAMPTZ NOT NULL,
    dropoff_at           TIMESTAMPTZ NOT NULL,
    pickup_date          DATE NOT NULL,

    passenger_count      SMALLINT
        CHECK (passenger_count IS NULL OR passenger_count BETWEEN 0 AND 9),

    trip_distance        NUMERIC(10, 2) NOT NULL
        CHECK (trip_distance >= 0),

    pu_location_id       INTEGER NOT NULL CHECK (pu_location_id BETWEEN 1 AND 265),
    do_location_id       INTEGER NOT NULL CHECK (do_location_id BETWEEN 1 AND 265),

    payment_type         SMALLINT
        CHECK (payment_type BETWEEN 0 AND 6),

    fare_amount          NUMERIC(10, 2),
    tip_amount           NUMERIC(10, 2),
    total_amount         NUMERIC(10, 2),

    store_and_fwd        BOOLEAN,

    CHECK (dropoff_at >= pickup_at)
);
"""

with engine.begin() as conn:
    conn.execute(text(ddl_clean))

In [73]:
insert_clean = """
WITH parsed AS (
    SELECT
        NULLIF(TRIM(vendor_id_raw), '')::SMALLINT                         AS vendor_id,
        NULLIF(TRIM(pickup_raw), '')::TIMESTAMP                           AS pickup_local,
        NULLIF(TRIM(dropoff_raw), '')::TIMESTAMP                          AS dropoff_local,
        NULLIF(TRIM(passenger_count_raw), '')::NUMERIC::SMALLINT          AS passenger_count,
        NULLIF(TRIM(trip_distance_raw), '')::NUMERIC(10, 2)               AS trip_distance,
        NULLIF(TRIM(pu_location_id_raw), '')::INTEGER                     AS pu_location_id,
        NULLIF(TRIM(do_location_id_raw), '')::INTEGER                     AS do_location_id,
        NULLIF(TRIM(payment_type_raw), '')::SMALLINT                      AS payment_type,
        NULLIF(TRIM(fare_amount_raw), '')::NUMERIC(10, 2)                 AS fare_amount,
        NULLIF(TRIM(tip_amount_raw), '')::NUMERIC(10, 2)                  AS tip_amount,
        NULLIF(TRIM(total_amount_raw), '')::NUMERIC(10, 2)                AS total_amount,
        CASE UPPER(TRIM(store_and_fwd_flag_raw))
            WHEN 'Y' THEN TRUE
            WHEN 'N' THEN FALSE
        END                                                               AS store_and_fwd
    FROM hw3_taxi_staging
)
INSERT INTO hw3_taxi_trips (
    vendor_id, pickup_at, dropoff_at, pickup_date,
    passenger_count, trip_distance, pu_location_id, do_location_id,
    payment_type, fare_amount, tip_amount, total_amount, store_and_fwd
)
SELECT
    vendor_id,
    pickup_local  AT TIME ZONE 'America/New_York',
    dropoff_local AT TIME ZONE 'America/New_York',
    pickup_local::DATE,
    passenger_count, trip_distance, pu_location_id, do_location_id,
    payment_type, fare_amount, tip_amount, total_amount, store_and_fwd
FROM parsed
WHERE pickup_local IS NOT NULL
  AND dropoff_local IS NOT NULL
  AND dropoff_local >= pickup_local;
"""

with engine.begin() as conn:
    result = conn.execute(text(insert_clean))
    print('inserted:', result.rowcount)

inserted: 99999


## 2. Staging і DDL: обґрунтування типів

**Staging (`hw3_taxi_staging`)**: усі 12 колонок мають тип `TEXT` з суфіксом `_raw`. Так `COPY` не падає на порожніх рядках, значеннях на кшталт `1.0` замість `1` чи некоректних датах. Типізація та очищення відбуваються окремим кроком через `INSERT INTO ... SELECT`.

**Clean (`hw3_taxi_trips`)**: типізована таблиця. Спільне правило очищення для всіх полів: `TRIM` → `NULLIF(..., '')` → приведення типу. Порожній рядок стає `NULL`, а не помилкою.

| Колонка | Семантика | Джерельний формат (Parquet → CSV) | Тип у clean | Правила очищення |
|---|---|---|---|---|
| `trip_id` | Технічний ключ поїздки. Природного унікального ID у джерелі немає. | — (генерується БД) | `BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY` | — |
| `vendor_id` | Постачальник TPEP-системи: 1 = Creative Mobile Technologies, 2 = Curb Mobility, 6 = Myle Technologies, 7 = Helix. | `int32` (`VendorID`) | `SMALLINT NOT NULL` | `::SMALLINT`. Без CHECK на список кодів, бо довідник постачальників змінюється з часом і жорсткий список зламав би завантаження нових місяців. |
| `pickup_at`, `dropoff_at` | Моменти посадки та висадки. | `datetime64[us]` без часової зони. Згідно з документацією TLC, це **локальний час Нью-Йорка**. | `TIMESTAMPTZ NOT NULL` | `::TIMESTAMP AT TIME ZONE 'America/New_York'`: сирий «годинник NY» явно локалізується в конкретну мить, яка зберігається в UTC. Зона задана назвою, а не зсувом `-05:00`, щоб правильно врахувати перехід на літній час. `NOT NULL`, бо поїздка без часу не має змісту. |
| `pickup_date` | Календарна дата посадки **за місцевим часом Нью-Йорка**. Використовується для групування за днями. | похідна від `tpep_pickup_datetime` | `DATE NOT NULL` | `pickup_local::DATE`, тобто беремо дату *до* переведення в UTC. `DATE` не має зони і не конвертується пізніше, тому сенс задаємо під час запису: аналітика ведеться за днями Нью-Йорка. Якщо брати UTC-дату, поїздки після 19:00 (UTC−5) потрапили б на наступний день. |
| `passenger_count` | Кількість пасажирів. Значення вводить водій. | `float64` (`1.0`, `2.0`, …), бо pandas зберігає цілі з пропусками як float. Є пропуски. | `SMALLINT`, `CHECK (passenger_count IS NULL OR passenger_count BETWEEN 0 AND 9)` | `::NUMERIC::SMALLINT`: двоступеневе приведення, бо `'1.0'::SMALLINT` дає помилку. `NULL` дозволено, бо це справжні пропуски джерела. `0` залишаємо для аудиту як доменну аномалію. |
| `trip_distance` | Відстань поїздки за таксометром, у **милях**. | `float64` | `NUMERIC(10,2) NOT NULL`, `CHECK (trip_distance >= 0)` | `::NUMERIC(10,2)`. Від'ємна відстань фізично неможлива. Нульову відстань залишаємо й досліджуємо в аудиті. |
| `pu_location_id`, `do_location_id` | Зони посадки та висадки за довідником TLC Taxi Zone. | `int32` (`PULocationID`, `DOLocationID`) | `INTEGER NOT NULL`, `CHECK (... BETWEEN 1 AND 265)` | `::INTEGER`. Межі 1–265 — це діапазон кодів у довіднику зон (264 і 265 означають невідому зону або зону поза NYC). |
| `payment_type` | Спосіб оплати: 0 = Flex Fare, 1 = картка, 2 = готівка, 3 = без оплати, 4 = спір, 5 = невідомо, 6 = анульована поїздка. | `int64` | `SMALLINT`, `CHECK (payment_type BETWEEN 0 AND 6)` | `::SMALLINT`. Код залишаємо числовим, як у джерелі. Розшифрування — справа довідника, а не fact-таблиці. |
| `fare_amount`, `tip_amount`, `total_amount` | Тариф за лічильником, чайові та загальна сума, у USD. | `float64` | `NUMERIC(10,2)` | `::NUMERIC(10,2)`: грошові суми з точністю до центів. `float` зберігає десяткові дроби наближено, і на сумах накопичуються похибки. **CHECK на ≥ 0 свідомо немає**: від'ємні суми — це реальні повернення чи коригування в джерелі, тож вони досліджуються в аудиті, а не відкидаються мовчки. |
| `store_and_fwd` | Чи зберігався запис у пам'яті таксі через відсутність зв'язку і чи надсилався пізніше. | `str`: `Y` / `N`, є пропуски | `BOOLEAN` | `CASE UPPER(TRIM(...)) WHEN 'Y' THEN TRUE WHEN 'N' THEN FALSE END`. Будь-що інше, зокрема `NULL`, стає `NULL`, бо значення не вигадуємо. |

**Обмеження рівня таблиці:** `CHECK (dropoff_at >= pickup_at)`, тобто поїздка не може закінчитися раніше, ніж почалася.

**Фільтрація під час перенесення:** `WHERE pickup_local IS NOT NULL AND dropoff_local IS NOT NULL AND dropoff_local >= pickup_local`. Умова дублює `NOT NULL` і `CHECK` на час. Без неї один некоректний рядок відхилив би весь `INSERT`: `CHECK` не фільтрує, а забороняє операцію. Усі інші аномалії, як-от від'ємні суми, нульова відстань чи 0 пасажирів, **не фільтруються**.

## 3. Data-quality audit

### 3.1. Контроль кількості рядків

In [74]:
pd.read_sql("""
SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_taxi_staging
UNION ALL
SELECT 'clean'   AS layer, COUNT(*) AS n_rows FROM hw3_taxi_trips
""", engine)

,layer,n_rows
0,staging,100000
1,clean,99999


In [75]:
# Рядок, який не потрапив у clean: висадка раніше за посадку
pd.read_sql("""
SELECT *
FROM hw3_taxi_staging
WHERE pickup_raw::TIMESTAMP > dropoff_raw::TIMESTAMP
""", engine)

,vendor_id_raw,pickup_raw,dropoff_raw,passenger_count_raw,trip_distance_raw,pu_location_id_raw,do_location_id_raw,payment_type_raw,fare_amount_raw,tip_amount_raw,total_amount_raw,store_and_fwd_flag_raw
0,1,2024-01-18 11:00:00,2024-01-18 10:57:07,1.0,1.7,181,89,1,19.5,0.0,21.0,N


**Висновок 3.1.** Staging містить 100 000 рядків, clean — 99 999. Відфільтровано рівно **1 рядок** за правилом `dropoff_local >= pickup_local` (воно ж — `CHECK (dropoff_at >= pickup_at)` у DDL): vendor 1, посадка `2024-01-18 11:00:00`, висадка `10:57:07`, тобто на ~3 хвилини *раніше* за посадку. Час посадки рівно `11:00:00` вказує на ручне введення або збій годинника таксометра. Інших фільтрів немає — від'ємні суми, нульова відстань, 0 пасажирів свідомо залишені в clean і аналізуються нижче.

### 3.2. Перевірка дублікатів

Business key: `(vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id)` — той самий постачальник, та сама мить посадки й висадки з точністю до секунди, ті самі зони. Гарантованого природного ключа в TLC немає, тому перевірка виявляє лише **потенційні** дублікати.

In [76]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id))
        AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id))
        AS duplicate_rows
FROM hw3_taxi_trips
""", engine)

,total_rows,distinct_business_keys,duplicate_rows
0,99999,99967,32


Знайдено 32 зайві рядки за business key. Подивимося, чим відрізняються рядки з однаковим ключем:

In [77]:
pd.read_sql("""
SELECT
    t.trip_id, t.vendor_id, t.pickup_at, t.dropoff_at,
    t.pu_location_id, t.do_location_id,
    t.payment_type, t.fare_amount, t.total_amount
FROM hw3_taxi_trips AS t
JOIN (
    SELECT vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id
    FROM hw3_taxi_trips
    GROUP BY vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id
    HAVING COUNT(*) > 1
) AS d
  USING (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id)
ORDER BY t.pickup_at, t.trip_id
LIMIT 100
""", engine)

,trip_id,vendor_id,pickup_at,dropoff_at,pu_location_id,do_location_id,payment_type,fare_amount,total_amount
0,769,2,2024-01-01 08:50:10+00:00,2024-01-01 09:04:50+00:00,230,143,4,17.0,22.00
1,770,2,2024-01-01 08:50:10+00:00,2024-01-01 09:04:50+00:00,230,143,4,-17.0,-22.00
2,5277,2,2024-01-03 07:32:50+00:00,2024-01-03 07:43:28+00:00,79,195,4,-21.2,-26.20
3,5278,2,2024-01-03 07:32:50+00:00,2024-01-03 07:43:28+00:00,79,195,4,21.2,26.20
4,11346,2,2024-01-05 03:16:02+00:00,2024-01-05 03:22:30+00:00,162,186,4,7.9,12.90
...,...,...,...,...,...,...,...,...,...
59,93378,2,2024-01-30 12:22:28+00:00,2024-01-30 12:22:32+00:00,28,28,3,3.0,4.50
60,94695,2,2024-01-30 19:50:52+00:00,2024-01-30 20:09:06+00:00,75,129,4,26.1,34.54
61,94696,2,2024-01-30 19:50:52+00:00,2024-01-30 20:09:06+00:00,75,129,4,-26.1,-34.54
62,95449,2,2024-01-30 23:07:31+00:00,2024-01-30 23:07:57+00:00,186,186,2,-3.0,-9.50


In [78]:
# Чи всі пари — це «поїздка + сторно» (сума пари = 0)?
pd.read_sql("""
WITH dup AS (
    SELECT
        vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
        COUNT(*)            AS n_rows,
        SUM(total_amount)   AS net_total,
        MIN(payment_type)   AS payment_type
    FROM hw3_taxi_trips
    GROUP BY vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id
    HAVING COUNT(*) > 1
)
SELECT
    payment_type,
    COUNT(*)                                AS n_pairs,
    COUNT(*) FILTER (WHERE net_total = 0)   AS pairs_net_zero,
    MAX(n_rows)                             AS max_rows_per_key
FROM dup
GROUP BY payment_type
ORDER BY n_pairs DESC
""", engine)

,payment_type,n_pairs,pairs_net_zero,max_rows_per_key
0,4,20,20,2
1,2,9,9,2
2,3,3,3,2


In [79]:
# Скільки всього рядків з від'ємною сумою і скільки з них мають «пару» у вибірці
pd.read_sql("""
SELECT
    COUNT(*) AS negative_rows,
    COUNT(*) FILTER (WHERE EXISTS (
        SELECT 1
        FROM hw3_taxi_trips AS p
        WHERE p.vendor_id      = n.vendor_id
          AND p.pickup_at      = n.pickup_at
          AND p.dropoff_at     = n.dropoff_at
          AND p.pu_location_id = n.pu_location_id
          AND p.do_location_id = n.do_location_id
          AND p.total_amount   = -n.total_amount
    )) AS negative_rows_with_pair_in_sample
FROM hw3_taxi_trips AS n
WHERE n.total_amount < 0
""", engine)

,negative_rows,negative_rows_with_pair_in_sample
0,1245,32


**Висновок 3.2.** За business key знайдено **32 пари** (64 рядки, по 2 на ключ), повних дублікатів (усі колонки однакові) — 0. Усі пари мають однакову структуру: рядок із додатними сумами + рядок із **тими самими сумами зі знаком мінус**, тобто чиста сума пари = 0. Це не помилкове дублювання, а **сторно (reversal)** — бухгалтерське скасування поїздки: `payment_type` 4 «спір» (20 пар), 2 «готівка» (9), 3 «без оплати» (3); усі — vendor 2.

- **Видаляти їх не можна**: прибравши лише від'ємний рядок, ми завищимо виручку; прибравши обидва — втратимо факт спірної поїздки.
- Для ML-аналізу поїздок (тривалість, попит) сторно-рядки слід фільтрувати (`total_amount < 0`) або згортати пару в одну поїздку з прапорцем `is_reversed`.
- **Побічний ефект вибірки:** від'ємних рядків у clean ~1 245, але «пару» у вибірці мають лише 32. Випадкова вибірка 100 тис. з ~3 млн розірвала більшість пар «поїздка + сторно» — у повному файлі таких пар значно більше. Тому суми по вибірці потрібно інтерпретувати обережно.

### 3.3. Null-rate і спеціальні маркери

#### 3.3.1. Справжні NULL у clean

In [80]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_rows,
    ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_passenger_count,
    ROUND(COUNT(*) FILTER (WHERE store_and_fwd   IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_store_and_fwd,
    ROUND(COUNT(*) FILTER (WHERE payment_type    IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_payment_type,
    ROUND(COUNT(*) FILTER (WHERE trip_distance   IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_trip_distance,
    ROUND(COUNT(*) FILTER (WHERE fare_amount     IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_fare_amount,
    ROUND(COUNT(*) FILTER (WHERE total_amount    IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_total_amount
FROM hw3_taxi_trips
""", engine)

,total_rows,pct_null_passenger_count,pct_null_store_and_fwd,pct_null_payment_type,pct_null_trip_distance,pct_null_fare_amount,pct_null_total_amount
0,99999,4.73,4.73,0.0,0.0,0.0,0.0


**Висновок 3.3.1.** Справжні `NULL` є лише у двох колонках: `passenger_count` і `store_and_fwd` — по **4.73%** (4 727 рядків). У `payment_type`, `trip_distance`, `fare_amount`, `total_amount` null-rate = **0%**. Однакова частка у двох колонках натякає, що це ті самі рядки — перевіряємо нижче.

#### 3.3.2. Спеціальні маркери — «прихований» пропуск

`NULL` — не єдиний спосіб записати «невідомо». У TLC пропуск часто закодовано валідним на вигляд значенням: `payment_type` 0 (Flex Fare) / 5 (Unknown), зони 264 / 265 (Unknown / Outside NYC), 0 пасажирів, нульова відстань.

In [81]:
pd.read_sql("""
SELECT
    COUNT(*) FILTER (WHERE passenger_count = 0)                       AS zero_passengers,
    COUNT(*) FILTER (WHERE trip_distance = 0)                         AS zero_distance,
    COUNT(*) FILTER (WHERE payment_type IN (0, 5))                    AS payment_flex_or_unknown,
    COUNT(*) FILTER (WHERE pu_location_id IN (264, 265))              AS pu_unknown_zone,
    COUNT(*) FILTER (WHERE do_location_id IN (264, 265))              AS do_unknown_zone,
    COUNT(*) FILTER (WHERE passenger_count IS NULL AND payment_type = 0) AS null_pc_and_payment_0
FROM hw3_taxi_trips
""", engine)

,zero_passengers,zero_distance,payment_flex_or_unknown,pu_unknown_zone,do_unknown_zone,null_pc_and_payment_0
0,1085,1991,4727,370,896,4727


**Висновок 3.3.2.**

- **`payment_type IN (0, 5)` — 4 727 рядків**, і всі вони мають код 0 (5 не трапляється). Колонка `null_pc_and_payment_0 = 4 727` доводить, що **всі NULL у `passenger_count` (і в `store_and_fwd`) — це рівно рядки з `payment_type = 0`**. Пропуски не випадкові (*missing not at random*): для Flex Fare-поїздок система просто не заповнює ці поля. Заповнювати їх середнім чи модою некоректно — краще окрема категорія або прапорець.
- **0 пасажирів — 1 085 рядків** (~1.1%): поїздка без пасажира не має доменного змісту, ймовірно водій не ввів значення. Це «прихований NULL» усередині валідного числа.
- **Нульова відстань — 1 991 рядок** (~2%): скасовані поїздки, посадка-висадка на місці або збій GPS/таксометра.
- **Невідома зона:** посадка — 370, висадка — 896. Висадок «поза NYC / невідомо» більше, ніж посадок — логічно, бо жовте таксі забирає пасажирів у місті, але може везти за його межі.

#### 3.3.3. Маркери в сирих даних (staging)

In [82]:
pd.read_sql("""
SELECT
    COUNT(*) FILTER (WHERE passenger_count_raw IS NULL)                         AS pc_null,
    COUNT(*) FILTER (WHERE TRIM(passenger_count_raw) = '')                      AS pc_empty,
    COUNT(*) FILTER (WHERE LOWER(TRIM(passenger_count_raw)) IN ('?', 'unknown', 'nan', 'null', 'none')) AS pc_text_markers,
    COUNT(*) FILTER (WHERE store_and_fwd_flag_raw IS NOT NULL
                       AND UPPER(TRIM(store_and_fwd_flag_raw)) NOT IN ('Y', 'N'))  AS flag_unexpected
FROM hw3_taxi_staging
""", engine)

,pc_null,pc_empty,pc_text_markers,flag_unexpected
0,4727,0,0,0


**Висновок 3.3.3.** У staging `passenger_count_raw` має 4 727 справжніх `NULL` і **0** порожніх рядків чи текстових маркерів (`?`, `unknown`, `NaN`, `null`); `store_and_fwd_flag_raw` не містить нічого, крім `Y`/`N`/`NULL`. Причина — джерело типізований Parquet: pandas записав пропуски як порожні поля CSV, а `COPY ... FORMAT csv` трактує їх як `NULL`. На відміну від CSV-датасетів UCI (`?` в Adult, `unknown` у Bank Marketing), тут текстових маркерів немає — пропуск закодовано або як `NULL`, або як службовий код у довідниковому полі.

**Підсумок аудиту.** Нульовий null-rate більшості колонок **не означає** якісних даних: у «повних» колонках сховані 0 пасажирів, нульові відстані, невідомі зони, сторно-рядки з від'ємними сумами та поїздка з висадкою раніше посадки. Рішення: у clean нічого не замінюємо й не видаляємо (крім фізично неможливої поїздки), аномалії позначаємо й аналізуємо в розділі 4.

## 4. Розподіли та потенційні аномалії

### 4.1. Розподіл категоріальної колонки: `payment_type`

In [83]:
pd.read_sql("""
SELECT
    payment_type,
    CASE payment_type
        WHEN 0 THEN 'Flex Fare'
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
    END AS payment_label,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_taxi_trips
GROUP BY payment_type
ORDER BY n_rows DESC, payment_type
LIMIT 10
""", engine)

,payment_type,payment_label,n_rows,pct_rows
0,1,Credit card,78131,78.13
1,2,Cash,14877,14.88
2,0,Flex Fare,4727,4.73
3,4,Dispute,1594,1.59
4,3,No charge,670,0.67


**Висновок 4.1.** Розподіл способів оплати сильно **незбалансований**: оплата карткою (`1`) — **~78%** поїздок, готівка (`2`) — **~15%**, разом це понад 93%. Далі йде Flex Fare (`0`) — **~4.7%**, і дві рідкісні категорії: спір (`4`) — **~1.6%** та без оплати (`3`) — **~0.7%**. Коди `5` (Unknown) і `6` (Voided trip) є в довіднику TLC, але у вибірці не трапляються — це результат, а не помилка.

- **Для ML:** якщо прогнозувати спосіб оплати, «наївна» модель, яка завжди відповідає «картка», отримала б ~78% accuracy, нічого не навчившись. Тому для такої задачі потрібні метрики, чутливі до дисбалансу (precision/recall, F1 за класами), або балансування класів.
- **Рідкісні категорії `3` і `4` важливі попри малу частку:** саме в них зосереджені сторно-пари з від'ємними сумами (аудит 3.2). Їх не можна відкидати як «шум» — вони несуть інформацію про спори й повернення.
- **Flex Fare (`0`)** — окремий тип записів із незаповненими `passenger_count` і `store_and_fwd` (аудит 3.3.2), тому в аналізі пасажирів його слід розглядати окремо.

### 4.2. Базова статистика числової колонки: `trip_distance`

In [84]:
pd.read_sql("""
SELECT
    COUNT(trip_distance)                                          AS n_non_null,
    MIN(trip_distance)                                            AS min_value,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY trip_distance)   AS p25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY trip_distance)   AS median,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY trip_distance)   AS p75,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance)   AS p99,
    MAX(trip_distance)                                            AS max_value,
    ROUND(AVG(trip_distance)::NUMERIC, 2)                         AS avg_value,
    ROUND(STDDEV_SAMP(trip_distance)::NUMERIC, 2)                 AS sd_value
FROM hw3_taxi_trips
""", engine)

,n_non_null,min_value,p25,median,p75,p99,max_value,avg_value,sd_value
0,99999,0.0,1.0,1.68,3.1,19.9,99.0,3.21,4.33


**Висновок 4.2.** `trip_distance` заповнена для всіх 99 999 поїздок, діапазон — від **0** до **99** миль. Ключове спостереження — **середнє (~3.21 милі) майже вдвічі більше за медіану (~1.68 милі)**, а стандартне відхилення (~4.33) **більше за саме середнє**. Це ознаки сильної **правобічної асиметрії**: більшість поїздок короткі (половина — до ~1.7 милі, три чверті — до ~3.1 милі, типові поїздки по Мангеттену), а рідкісні довгі поїздки (аеропорти JFK / LaGuardia, ~15–20 миль, і далі за місто) утворюють довгий «хвіст» праворуч і тягнуть середнє вгору.

Висновки для аналізу:
- «Типову» поїздку описує **медіана**, а не середнє: середнє 3.21 милі не відповідає поведінці більшості пасажирів.
- Для опису розкиду краще підходять **перцентилі** (p25–p75, p99), ніж пара «середнє ± SD»: при SD > середнього межа «середнє − 1σ» виходить від'ємною (≈ −1.1 милі), що фізично неможливо.
- Для ML-моделей, чутливих до масштабу, `trip_distance` варто логарифмувати (`log(1 + x)`), щоб зменшити вплив хвоста.

### 4.3. Outlier-check: доменна перевірка

**Чому доменна перевірка, а не правило 3σ.** Правило 3σ ґрунтується на властивостях **нормального (симетричного)** розподілу: для нього за межі «середнє ± 3σ» виходить лише ~0.3% значень, тож такі значення справді рідкісні й підозрілі. Але в 4.2 показано, що `trip_distance` сильно асиметрична (середнє ≈ 2 × медіана, SD > середнього). Для неї правило 3σ дає межу ≈ 3.21 + 3 · 4.33 ≈ **16.2 милі** і позначає як «викиди» близько **4% поїздок** — у ~13 разів більше, ніж очікувалося б для нормального розподілу. Серед них — звичайні поїздки в аеропорти, тобто правило ловить не помилки, а легітимну частину хвоста. Аналогічно асиметричні й грошові колонки (`fare_amount`).

Тому обрано **доменну перевірку**: шукаємо записи, що суперечать фізиці чи бізнес-логіці таксі. Пороги взято з предметної області, а не зі статистики:
- `fare_amount < 0`, `tip_amount < 0` — від'ємні суми (сторно / повернення);
- тривалість > **3 годин** — нереалістично для поїздки містом;
- середня швидкість > **80 миль/год** — неможливо в міському трафіку Нью-Йорка;
- відстань > **50 миль** — рідкісні поїздки далеко за місто, потребують перевірки;
- дата посадки поза січнем 2024 — запис з іншого місяця у місячному файлі.

In [85]:
pd.read_sql("""
WITH trips AS (
    SELECT
        *,
        EXTRACT(EPOCH FROM (dropoff_at - pickup_at)) / 60.0 AS duration_min
    FROM hw3_taxi_trips
)
SELECT
    COUNT(*) FILTER (WHERE fare_amount < 0)                          AS negative_fare,
    COUNT(*) FILTER (WHERE tip_amount < 0)                           AS negative_tip,
    COUNT(*) FILTER (WHERE duration_min > 180)                       AS longer_than_3h,
    COUNT(*) FILTER (WHERE duration_min > 0
                       AND trip_distance / (duration_min / 60.0) > 80) AS speed_over_80mph,
    COUNT(*) FILTER (WHERE trip_distance > 50)                       AS distance_over_50mi,
    COUNT(*) FILTER (WHERE pickup_date NOT BETWEEN DATE '2024-01-01'
                                               AND DATE '2024-01-31') AS pickup_outside_month
FROM trips
""", engine)

,negative_fare,negative_tip,longer_than_3h,speed_over_80mph,distance_over_50mi,pickup_outside_month
0,1306,5,47,35,8,2


In [86]:
pd.read_sql("""
SELECT
    trip_id, vendor_id,
    pickup_at AT TIME ZONE 'America/New_York' AS pickup_ny,
    ROUND(EXTRACT(EPOCH FROM (dropoff_at - pickup_at)) / 60.0, 1) AS duration_min,
    trip_distance, fare_amount, total_amount
FROM hw3_taxi_trips
WHERE (dropoff_at - pickup_at) > INTERVAL '3 hours'
   OR (EXTRACT(EPOCH FROM (dropoff_at - pickup_at)) > 0
       AND trip_distance / (EXTRACT(EPOCH FROM (dropoff_at - pickup_at)) / 3600.0) > 80)
ORDER BY duration_min DESC
LIMIT 20
""", engine)

,trip_id,vendor_id,pickup_ny,duration_min,trip_distance,fare_amount,total_amount
0,87333,2,2024-01-28 00:30:07,1439.1,0.61,7.2,14.03
1,37147,2,2024-01-13 03:55:58,1438.5,2.26,13.5,18.50
2,74767,2,2024-01-24 17:19:01,1437.8,1.65,12.8,21.23
3,63888,2,2024-01-21 02:33:23,1437.6,0.82,7.2,14.64
4,21964,2,2024-01-08 15:10:43,1435.5,1.40,9.3,13.30
5,83307,2,2024-01-26 23:04:09,1435.3,6.99,35.2,48.24
6,53536,2,2024-01-18 09:54:35,1434.8,1.66,12.1,13.60
7,51782,2,2024-01-17 18:31:12,1434.5,1.49,13.5,20.00
8,84050,2,2024-01-27 04:21:38,1434.1,15.87,59.7,67.94
9,17057,2,2024-01-06 17:03:33,1434.0,5.71,28.2,35.42


**Висновок 4.3.** Доменна перевірка знайшла кілька груп аномалій. Найпомітніша — поїздки тривалістю **~1439 хвилин (23 год 59 хв)** зі звичайною відстанню в кілька миль і звичайним тарифом: це класичний збій TLC, коли таксометр не закрили вчасно. Фізично неможливі швидкості > 80 миль/год вказують на помилки GPS або часових міток. Жодну аномалію не видалено з clean: рішення залежить від задачі, тому пропонуємо дії.

| Аномалія | Кількість | Ймовірна причина | Дія |
|---|---|---|---|
| `fare_amount < 0` | ~1 306 | сторно / повернення (пари з аудиту 3.2), `payment_type` 3–4 | **додати прапорець** `is_reversal`; **виключити з training data** для моделей тарифу й тривалості; для фінансових звітів — залишити |
| `tip_amount < 0` | 5 | коригування чайових у сторно-рядку | **додати прапорець** разом із `is_reversal` |
| тривалість > 3 год (до ~24 год) | ~47 | таксометр не закрито вчасно, збій часу | **виключити з training data** для моделей тривалості; **передати на перевірку джерела** (vendor) |
| швидкість > 80 миль/год | ~35 | помилка GPS або часових міток (дуже коротка тривалість при значній відстані) | **додати прапорець** `is_suspicious_speed`; **виключити з training data** |
| відстань > 50 миль | ~8 | рідкісні далекі поїздки за місто (Лонг-Айленд, Нью-Джерсі тощо) | **залишити як валідний рідкісний випадок**, якщо тариф і тривалість узгоджуються з відстанню; інакше — прапорець |
| `pickup_date` поза січнем | ~2 | поїздки, що почалися 31.12 до опівночі й завершилися в січні; TLC відносить поїздку до місяця завершення | **залишити** — це коректна поведінка джерела; у місячній аналітиці фільтрувати за `pickup_date`, якщо потрібен «чистий» січень |

Великі значення самі по собі **не є підставою для видалення**: поїздка на 50 миль в аеропорт велика, але справжня. Виключаються з навчання лише записи, що суперечать фізиці чи бізнес-логіці, і навіть вони залишаються в clean із прапорцем — щоб не втратити інформацію про якість джерела.

## Завдання 5. Десять DQL/EDA-запитів

### Підготовка: довідник зон TLC

Основна таблиця містить лише числові коди зон. Для запитів з `ILIKE` і назвами районів завантажуємо офіційний довідник
[Taxi Zone Lookup](https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv) (265 рядків).
Колонки перейменовуємо в snake_case, щоб у SQL не брати назви в лапки.

In [87]:
zones = pd.read_csv('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv')
zones.columns = ['location_id', 'borough', 'zone', 'service_zone']
zones.to_sql('hw3_taxi_zones', engine, if_exists='replace', index=False)
zones.tail(3)

,location_id,borough,zone,service_zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,NaN,NaN
264,265,NaN,Outside of NYC,NaN


### Q1. У які години найбільше поїздок?

In [88]:
pd.read_sql("""
SELECT
    EXTRACT(HOUR FROM pickup_at AT TIME ZONE 'America/New_York')::INT AS pickup_hour,
    COUNT(*) AS n_trips
FROM hw3_taxi_trips
GROUP BY pickup_hour
ORDER BY n_trips DESC
LIMIT 5;
""", engine)

,pickup_hour,n_trips
0,18,7082
1,17,6946
2,16,6487
3,15,6476
4,19,6290


Маємо діапазон годин з 16 по 19. Це вечірній час - повернення з роботи.

### Q2. Чи залежать чайові від способу оплати?

In [89]:
pd.read_sql("""
SELECT
    payment_type,
    COUNT(*) AS n_trips,
    ROUND(AVG(tip_amount), 2) AS avg_tip,
    ROUND(AVG(tip_amount / NULLIF(fare_amount, 0)) * 100, 2) AS avg_tip_pct_of_fare
FROM hw3_taxi_trips
WHERE payment_type IN (1, 2)
  AND fare_amount > 0
GROUP BY payment_type
ORDER BY payment_type;
""", engine)

,payment_type,n_trips,avg_tip,avg_tip_pct_of_fare
0,1,78125,4.15,25.8
1,2,14592,0.00,0.0


Картка — у середньому $4.15 (~25.8% від тарифу), готівка — рівно $0.00.

Це не означає, що пасажири з готівкою не дають чайових. Готівкові чайові система просто не фіксує. Це обмеження джерела, а не поведінка людей. Якщо тренувати модель «хто дає чайові», вона вивчить артефакт збору даних.

### Q3. Скільки поїздок починається в аеропортах і наскільки вони дорожчі?

In [92]:
pd.read_sql(text("""
SELECT
    z.zone,
    COUNT(*) AS n_pickups,
    ROUND(AVG(t.trip_distance), 2) AS avg_distance,
    ROUND(AVG(t.total_amount), 2) AS avg_total
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS z ON z.location_id = t.pu_location_id
WHERE z.zone ILIKE '%airport%'
GROUP BY z.zone
ORDER BY n_pickups DESC
"""), engine)

,zone,n_pickups,avg_distance,avg_total
0,JFK Airport,4819,15.31,76.24
1,LaGuardia Airport,2940,9.65,64.76
2,Newark Airport,9,0.00,102.63


JFK — 4819 (у середньому 15.3 милі, $76), LaGuardia — 2940 (9.7 милі, $65), Newark — 9 поїздок із середньою відстанню 0.00 при $103.

Newark — аномалія. Аеропорт у Нью-Джерсі, жовте таксі там рідко бере пасажирів, а нульова відстань при сумі $100+ схожа на помилку таксометра чи фіксований тариф. Це кандидат на перевірку джерела.

### Q4. Скільки пасажирів зазвичай у поїздці, включно з невідомими?

In [93]:
pd.read_sql("""
SELECT
    COALESCE(passenger_count::TEXT, 'unknown') AS passengers,
    COUNT(*) AS n_trips,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_trips
FROM hw3_taxi_trips
GROUP BY passengers
ORDER BY n_trips DESC;
""", engine)

,passengers,n_trips,pct_trips
0,1,73738,73.74
1,2,13809,13.81
2,unknown,4727,4.73
3,3,3071,3.07
4,4,1722,1.72
5,5,1140,1.14
6,0,1085,1.09
7,6,705,0.71
8,8,2,0.00


Результат: 1 пасажир — 73.7%, 2 — 13.8%, unknown — 4.7%, далі 3–6 пасажирів, 0 — 1.1%, і 8 пасажирів — двічі.

Таксі — це переважно поїздки самому. 8 пасажирів у жовтому таксі фізично малоймовірно (максимум 4–5, у мінівені 6), тож це підозріло.

### Q5. Скільки поїздок не є одиночними?

In [96]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_trips,
    COUNT(*) FILTER (WHERE passenger_count <> 1)                AS not_single_ne,
    COUNT(*) FILTER (WHERE passenger_count IS DISTINCT FROM 1)  AS not_single_distinct,
    COUNT(*) FILTER (WHERE passenger_count IS NULL)             AS unknown_passengers
FROM hw3_taxi_trips;
""", engine)

,total_trips,not_single_ne,not_single_distinct,unknown_passengers
0,99999,21534,26261,4727


Результат: <> дає 21 534, IS DISTINCT FROM — 26 261. Різниця 4727, рівно кількість NULL.

Звичайне <> мовчки загубило майже 5 тис. рядків. У кожному звіті з умовою != по колонці з NULL прихована така помилка.

### Q6. Які постачальники й способи оплати дають записи без кількості пасажирів?

In [97]:
pd.read_sql("""
SELECT DISTINCT
    vendor_id,
    payment_type
FROM hw3_taxi_trips
WHERE passenger_count IS NULL
ORDER BY vendor_id, payment_type;
""", engine)

,vendor_id,payment_type
0,1,0
1,2,0
2,6,0


У кожного постачальника NULL трапляється тільки з payment_type = 0. Отже, причина не в конкретному постачальнику, а в самому типі тарифу Flex Fare. Це підтверджує висновок аудиту 3.3.2 з іншого боку.

### Q7. Які поїздки мають тариф, що не відповідає відстані?

In [98]:
pd.read_sql("""
SELECT
    trip_id,
    pickup_at AT TIME ZONE 'America/New_York' AS pickup_ny,
    trip_distance,
    fare_amount,
    payment_type
FROM hw3_taxi_trips
WHERE (
        (trip_distance > 10 AND fare_amount BETWEEN 0 AND 20)
     OR (trip_distance = 0 AND fare_amount > 50)
      )
  AND NOT payment_type IN (3, 4)
ORDER BY fare_amount DESC
LIMIT 10;
""", engine)

,trip_id,pickup_ny,trip_distance,fare_amount,payment_type
0,90441,2024-01-29 01:54:38,0.0,215.00,2
1,51042,2024-01-17 15:36:28,0.0,200.22,2
2,67281,2024-01-22 12:24:44,0.0,168.00,1
3,49844,2024-01-17 09:27:37,0.0,168.00,1
4,21401,2024-01-08 11:19:55,0.0,165.00,1
5,19594,2024-01-07 14:48:15,0.0,151.00,1
6,20769,2024-01-08 01:34:17,0.0,150.00,1
7,16793,2024-01-06 15:53:00,0.0,150.00,1
8,8836,2024-01-04 11:57:08,0.0,150.00,1
9,63180,2024-01-20 22:36:37,0.0,150.00,1


Логіка: «довга, але дешева» або «нульова відстань, але дорога», і при цьому не спір чи безоплатна поїздка, бо там суми й так особливі.
Результат: усього 274 поїздки. У топ-10 усі мають 0 миль при тарифі $150–215.
Нульова відстань при тарифі $150+ — схоже на фіксовані тарифи (наприклад, в аеропорт, де лічильник не вмикали) або на збій GPS.

### Q8. Як змінювалася кількість поїздок у перші два тижні січня?

In [99]:
pd.read_sql("""
SELECT
    pickup_date,
    TO_CHAR(pickup_date, 'Dy') AS weekday,
    COUNT(*) AS n_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance
FROM hw3_taxi_trips
WHERE pickup_date BETWEEN DATE '2024-01-01' AND DATE '2024-01-14'
GROUP BY pickup_date
ORDER BY pickup_date;
""", engine)

,pickup_date,weekday,n_trips,avg_distance
0,2024-01-01,Mon,2716,4.15
1,2024-01-02,Tue,2497,4.12
2,2024-01-03,Wed,2705,3.74
3,2024-01-04,Thu,3661,3.23
4,2024-01-05,Fri,3477,3.11
5,2024-01-06,Sat,3355,3.13
6,2024-01-07,Sun,2306,3.81
7,2024-01-08,Mon,2628,3.35
8,2024-01-09,Tue,3156,2.81
9,2024-01-10,Wed,3211,3.28


Результат: 1–2 січня поїздок менше (~2500–2700), а середня відстань більша (~4.1 милі). З 4 січня ~3500 поїздок на день. Неділі (7 і 14 січня) теж слабші.

Свята: 1 січня — вихідний, а 2 січня багато хто ще не вийшов на роботу. Менше коротких робочих поїздок по Мангеттену, тож середня відстань росте. Для прогнозування попиту свята й дні тижня — важливі ознаки.

### Q9. Які маршрути між різними зонами найпопулярніші?

In [100]:
pd.read_sql("""
SELECT
    pz.zone AS pickup_zone,
    dz.zone AS dropoff_zone,
    COUNT(*) AS n_trips,
    ROUND(AVG(t.total_amount), 2) AS avg_total
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS pz ON pz.location_id = t.pu_location_id
JOIN hw3_taxi_zones AS dz ON dz.location_id = t.do_location_id
WHERE t.pu_location_id IS DISTINCT FROM t.do_location_id
GROUP BY pz.zone, dz.zone
ORDER BY n_trips DESC
LIMIT 10;
""", engine)

,pickup_zone,dropoff_zone,n_trips,avg_total
0,Upper East Side South,Upper East Side North,703,15.86
1,Upper East Side North,Upper East Side South,639,15.75
2,Midtown Center,Upper East Side South,325,16.54
3,Upper West Side South,Upper West Side North,308,13.68
4,Upper East Side South,Midtown Center,294,16.04
5,Upper West Side South,Lincoln Square East,292,14.31
6,Lincoln Square East,Upper West Side South,282,15.13
7,Midtown Center,Upper East Side North,279,21.23
8,Lenox Hill West,Upper East Side North,274,15.64
9,Upper East Side North,Midtown Center,265,20.56


Результат: лідер — Upper East Side South ↔ North (703 і 639 поїздок), далі Midtown ↔ Upper East Side і Upper West Side. Середній чек $14–21.
Топ-маршрути — це короткі поїздки між сусідніми районами Мангеттена. Ядро бізнесу жовтого таксі — внутрішньоміські поїздки на 1–2 милі, а не аеропорти.

### Q10. Який постачальник дає найбільше проблемних записів?

In [101]:
pd.read_sql("""
SELECT
    vendor_id,
    COUNT(*) AS n_trips,
    ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0 / COUNT(*), 2) AS pct_null_passengers,
    ROUND(COUNT(*) FILTER (WHERE passenger_count = 0)     * 100.0 / COUNT(*), 2) AS pct_zero_passengers,
    ROUND(COUNT(*) FILTER (WHERE trip_distance = 0)       * 100.0 / COUNT(*), 2) AS pct_zero_distance,
    ROUND(COUNT(*) FILTER (WHERE total_amount < 0)        * 100.0 / COUNT(*), 2) AS pct_negative_total
FROM hw3_taxi_trips
GROUP BY vendor_id
ORDER BY n_trips DESC;
""", engine)

,vendor_id,n_trips,pct_null_passengers,pct_zero_passengers,pct_zero_distance,pct_negative_total
0,2,75209,4.12,0.00,1.19,1.66
1,1,24784,6.55,4.37,4.42,0.00
2,6,6,100.00,0.00,0.00,0.00


Проблеми якості залежать від постачальника:

0 пасажирів і нульові відстані дає майже лише vendor 1. Схоже, його термінал записує 0, коли водій нічого не ввів.

Від'ємні суми (сторно) є лише у vendor 2. Отже, лише його система записує повернення окремими рядками, а vendor 1 робить це якось інакше.

Vendor 6 — це 6 поїздок, і всі Flex Fare без кількості пасажирів.

## Завдання 6. Reflection

**Проблеми якості.** Справжні NULL є лише в `passenger_count` і `store_and_fwd` (4.73%), і всі вони припадають на `payment_type = 0` (Flex Fare) — пропуски не випадкові, а залежать від типу тарифу. Значно більше проблем приховано в «повних» колонках: 0 пасажирів, нульова відстань, службові зони 264/265, поїздки тривалістю ~24 години, швидкість понад 80 миль/год. Дублікати за business key виявилися сторно-парами з від'ємними сумами. Важливо, що аномалії залежать від постачальника: нулі дає переважно vendor 1, сторно — лише vendor 2.

**Features.** Використав би годину, день тижня та свята (за місцевим часом NY), зони посадки й висадки, `vendor_id`, `trip_distance` після log-перетворення через сильну асиметрію. Не використав би `store_and_fwd` — це технічна ознака передачі даних, а `tip_amount` — лише для карткових оплат, бо готівкові чайові система не фіксує.

**Leakage.** Якщо прогнозувати вартість або тривалість до початку поїздки, `fare_amount`, `tip_amount`, `total_amount`, `dropoff_at` і фактична `trip_distance` стають відомими лише після її завершення. Їх треба виключити або замінити оцінками (наприклад, відстанню за маршрутом).

**Pipeline.** Порадив би додати унікальний `trip_id` у джерело, зв'язувати сторно з оригінальною поїздкою, відхиляти записи з висадкою раніше посадки, передавати час в UTC або явно з часовою зоною й валідувати дані окремо для кожного постачальника.

**ML-задача.** Природно підходять регресія (тариф, тривалість, чайові) та forecasting попиту за зонами й годинами.